## Notebook 05d -- Forward-only subject-dependent check (M3)
Rung 1 (subject-dependent) tests wear `j` with a model trained on **all other wears** of the same participant, before and after `j`. A real user has only the wears before `j`. This notebook retrains rung 1 forward only: expanding window, test wear `j = 2..K`, train on wears `1..j-1`. Plan: `docs/logs/20261012_145403_forward_only_rung1_plan.md`.
Hyperparameters are **not** searched again. Each forward fold reuses the stored rung-1 `best_params` of the fold with the same participant and the same test session. Disclose: those parameters were tuned with wears from both sides of `j`.
Models: Ridge, XGBoost, MLP, same seed and preprocessing as 03 / 04 / 05. Output: `fold_metrics_forward_sd.json` (resumable). This notebook only produces numbers. Statistics are a later phase, in `significance_analysis.py`.
Known confound: the forward model has fewer wears (1 to K-1) than rung 1 (K-1). No size control (plan decision 10).
Run headless from the `notebooks/` folder: `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 05d_forward_subject_dependent.ipynb`

Load the shared data (same `FEATURES`, `FEATURE_COLS`, `SESSION_FOLDS`, `MANIFEST` as 03/04/05).

In [ ]:
from _common import *

Imports and device. Same stack as `05c_rung4_weighted_refit.ipynb`.

In [ ]:
import gc
import os
import re
import time
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

Switches and paths. `RUN_LOOP = False` stops after the reproduction check and the timing cell. `SMOKE = True` writes to a separate file and runs one fold per model (participant 10, test wear index 1).

In [ ]:
RUN_LOOP = True
SMOKE = False

OUT_PATH = CURRENT_FOLDER / ('fold_metrics_forward_sd_smoke.json' if SMOKE else 'fold_metrics_forward_sd.json')
RUNG1_FILES = {'Ridge': 'fold_metrics_ridge.json', 'XGBoost': 'fold_metrics_xgboost.json', 'MLP': 'fold_metrics_mlp.json'}
print('RUN_LOOP:', RUN_LOOP, '| SMOKE:', SMOKE, '->', OUT_PATH.name)

### Task 0.1 and 0.2 -- session order
Session numbers must be in time order for every participant. The check reads the folder timestamps `recordings/participantNNN_<ts>/sessionN_<ts>/` and compares the time order with the session numbers in the manifest. Participant 001 has no session 4 and participant 010 has no session 6, so the session lists come from the manifest, never from `range(1, 11)`.

In [ ]:
SESSIONS = {int(p.removeprefix('participant')): sorted(v['sessions']) for p, v in MANIFEST['participants'].items()}

for p, sess in SESSIONS.items():
    stamps = []
    for s in sess:
        s_dir = session_dir(p, s)
        m = re.fullmatch(r'session(\d+)_(\d{8}_\d{6})', s_dir.name)
        assert m is not None and int(m.group(1)) == s, (p, s, s_dir.name)
        stamps.append(m.group(2))
    assert stamps == sorted(stamps) and len(set(stamps)) == len(stamps), (p, sess, stamps)   # time order == numbering
    assert sess == MANIFEST['participants'][f'participant{p:03d}']['sessions']                  # manifest list is already in order

assert SESSIONS[1] == [1, 2, 3, 5, 6, 7, 8, 9, 10]      # participant 001: 9 sessions, session 4 excluded
assert SESSIONS[10] == [1, 2, 3, 4, 5, 7, 8, 9, 10]     # participant 010: 9 sessions, session 6 excluded
n_sessions = {p: len(s) for p, s in SESSIONS.items()}
assert sum(n_sessions.values()) == 98
assert sum(n - 1 for n in n_sessions.values()) == 88    # forward folds per model
print('sessions per participant:', n_sessions)
print('total sessions:', sum(n_sessions.values()), '| forward folds per model:', sum(n - 1 for n in n_sessions.values()))

Model builders, copied from `05c_rung4_weighted_refit.ipynb` (which copied 03 / 04 / 05). The MLP `fit` has no `sample_weight` here because no weights are used.

In [ ]:
def make_ridge(params):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=params['alpha'])),
    ])


def make_xgboost(params):
    return XGBRegressor(
        **params,
        random_state=RANDOM_SEED,
        device='cuda',
        tree_method='hist',
        n_jobs=-1,
    )


class MLPTorch(nn.Module):
    def __init__(self, n_features, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPRegressor:
    def __init__(self, hidden_dims, dropout, lr, weight_decay, batch_size, n_epochs, random_state=RANDOM_SEED):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        self.model_ = MLPTorch(X.shape[1], self.hidden_dims, self.dropout).to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()

        X_t = torch.from_numpy(X).to(DEVICE)
        y_t = torch.from_numpy(y).to(DEVICE)
        n = len(X_t)

        self.model_.train()
        for epoch in range(self.n_epochs):
            perm = torch.randperm(n, device=DEVICE)
            for start in range(0, n, self.batch_size):
                idx = perm[start:start + self.batch_size]
                opt.zero_grad()
                pred = self.model_(X_t[idx])
                loss = loss_fn(pred, y_t[idx])
                loss.backward()
                opt.step()
        del X_t, y_t
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        device = next(self.model_.parameters()).device
        X_t = torch.from_numpy(X).to(device)
        self.model_.eval()
        with torch.no_grad():
            pred = self.model_(X_t)
        return pred.cpu().numpy()


def hidden_dims_from(width, n_layers):
    dims = []
    w = width
    for _ in range(n_layers):
        dims.append(max(w, 8))
        w = max(w // 2, 8)
    return dims


def make_mlp(params):
    hidden_dims = hidden_dims_from(params['width'], params['n_layers'])
    return Pipeline([
        ('scale', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_dims=hidden_dims,
            dropout=params['dropout'],
            lr=params['lr'],
            weight_decay=params['weight_decay'],
            batch_size=params['batch_size'],
            n_epochs=params['n_epochs'],
            random_state=RANDOM_SEED,
        )),
    ])


MODELS = {'Ridge': make_ridge, 'XGBoost': make_xgboost, 'MLP': make_mlp}
print(list(MODELS))

Load the stored rung-1 results. They give the `best_params` for each forward fold and the numbers to compare with. Key: `(participant, test_session)`.

In [ ]:
RUNG1 = {}
for model_name, json_name in RUNG1_FILES.items():
    with open(CURRENT_FOLDER / json_name) as f:
        RUNG1[model_name] = {(r['participant_id'], r['test_session']): r for r in json.load(f)}
    assert len(RUNG1[model_name]) == 98, model_name
    assert set(RUNG1[model_name]) == {(p, s) for p, sess in SESSIONS.items() for s in sess}
print({m: len(v) for m, v in RUNG1.items()})

### Task 1.2 -- reproduction check
Retrain one current rung-1 fold (all other wears, stored `best_params`) and compare with the stored RMSE to 4 decimals. Ridge and XGBoost must match, or the run stops. The MLP is trained twice, to show whether it is deterministic. It also stops the run if it does not match.
The fold is participant 002, test session 5, an interior wear, so training rows come from wears before and after it.

In [ ]:
REPRO_P, REPRO_S = 2, 5
repro_fold = next(f for f in SESSION_FOLDS if f['participant_id'] == REPRO_P and f['test_session'] == REPRO_S)
assert repro_fold['train_sessions'] == [s for s in SESSIONS[REPRO_P] if s != REPRO_S]   # all other wears

held_mask = FEATURES['participant_id'] == REPRO_P
train_mask = held_mask & FEATURES['session_id'].isin(repro_fold['train_sessions'])
test_mask = held_mask & (FEATURES['session_id'] == REPRO_S)
assert not (train_mask & test_mask).any()
X_train = FEATURES.loc[train_mask, FEATURE_COLS].to_numpy()
y_train = FEATURES.loc[train_mask, 'pct_mvc_corr'].to_numpy()
X_test = FEATURES.loc[test_mask, FEATURE_COLS].to_numpy()
y_test = FEATURES.loc[test_mask, 'pct_mvc_corr'].to_numpy()

repro = {}
for model_name, make_model in MODELS.items():
    stored = RUNG1[model_name][(REPRO_P, REPRO_S)]
    stored_rmse = stored['metrics']['overall']['rmse']
    tries = []
    for attempt in range(2 if model_name == 'MLP' else 1):
        model = make_model(stored['best_params'])
        model.fit(X_train, y_train)
        tries.append(float(np.sqrt(np.mean((np.asarray(model.predict(X_test), dtype=float) - y_test) ** 2))))
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    repro[model_name] = {'stored': stored_rmse, 'retrained': tries}
    ok = all(round(t, 4) == round(stored_rmse, 4) for t in tries)
    print(f"{model_name:8s} p{REPRO_P} s{REPRO_S}  stored={stored_rmse:.6f}  retrained={[round(t, 6) for t in tries]}  match to 4 decimals: {ok}")
    if not ok:
        raise RuntimeError(f'{model_name}: retrained rmse {tries} != stored rmse {stored_rmse:.6f}')
del X_train, y_train, X_test, y_test
gc.collect()

### Timing -- one forward fold per model at the largest training size
Largest training size = test wear `j = K`, so participant 002 (K = 10), test session 10, trained on sessions 1 to 9. Fit seconds only (predict excluded). The projection scales the measured seconds by training rows over all 88 folds. The upper bound uses the largest fold for all 88 folds. Run after the reproduction check, so CUDA is warm.

In [ ]:
TIME_P = 2
time_sessions = SESSIONS[TIME_P]
time_test = time_sessions[-1]
time_train = time_sessions[:-1]
assert len(time_train) == 9

t_held = FEATURES['participant_id'] == TIME_P
t_train_mask = t_held & FEATURES['session_id'].isin(time_train)
t_test_mask = t_held & (FEATURES['session_id'] == time_test)
assert not (t_train_mask & t_test_mask).any()
X_train = FEATURES.loc[t_train_mask, FEATURE_COLS].to_numpy()
y_train = FEATURES.loc[t_train_mask, 'pct_mvc_corr'].to_numpy()

# training rows of every forward fold, from the row counts per (participant, session)
rows_per = FEATURES.groupby(['participant_id', 'session_id']).size()
fold_rows = []
for p, sess in SESSIONS.items():
    for i in range(1, len(sess)):
        fold_rows.append(int(sum(rows_per[(p, s)] for s in sess[:i])))
assert len(fold_rows) == 88
n_rows_timed = len(X_train)
assert n_rows_timed == int(sum(rows_per[(TIME_P, s)] for s in time_train))
print('timed fold rows:', n_rows_timed, '| largest forward fold rows over all participants:', max(fold_rows))

timing = {}
for model_name, make_model in MODELS.items():
    params = RUNG1[model_name][(TIME_P, time_test)]['best_params']
    model = make_model(params)
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_s = time.time() - t0
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    projected_scaled_s = fit_s * sum(fold_rows) / n_rows_timed
    upper_s = fit_s * len(fold_rows)
    timing[model_name] = {'fit_s': fit_s, 'projected_88_scaled_s': projected_scaled_s, 'upper_88_s': upper_s}
    print(f"{model_name:8s} fit at {n_rows_timed} rows: {fit_s:6.1f} s | 88 folds scaled by rows: {projected_scaled_s / 60:6.1f} min | upper bound: {upper_s / 60:6.1f} min")
print(f"all 3 models (264 fits), scaled: {sum(t['projected_88_scaled_s'] for t in timing.values()) / 3600:.2f} h | upper bound: {sum(t['upper_88_s'] for t in timing.values()) / 3600:.2f} h")
del X_train, y_train
gc.collect()

### Forward folds, all models
Resumable. If the output file exists, load it and skip every `(model, participant, test_session)` already in it. The file is rewritten after every fold. For each participant, in time order: test wear `s_j` for `j = 2..K`, train on `s_1 .. s_(j-1)` only.
Each record has the same `metrics` structure as the `fold_metrics_*.json` files (`overall`, `by_phase`, `by_level`), plus `model`, `n_train_wears` and `train_sessions`. `params_from` names the rung-1 fold that supplied `best_params`.

In [ ]:
def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj


def _save_checkpoint(results):
    tmp_path = OUT_PATH.with_suffix('.json.tmp')          # write then rename, so an interrupt never leaves a half-written file
    with open(tmp_path, 'w') as f:
        json.dump(_json_safe(results), f, indent=2)
    os.replace(tmp_path, OUT_PATH)


if OUT_PATH.exists():
    with open(OUT_PATH) as f:
        fold_results = json.load(f)
else:
    fold_results = []

done_keys = {(r['model'], r['participant_id'], r['test_session']) for r in fold_results}
n_total = len(MODELS) * 88
print(f"resuming with {len(fold_results)} folds already done, {n_total - len(fold_results)} remaining (full run)")

In [ ]:
if RUN_LOOP:
    t_loop = time.time()
    for model_name, make_model in MODELS.items():
        for p_num, sess in SESSIONS.items():
            if SMOKE and p_num != 10:
                continue
            for j in range(1, len(sess)):                       # j is a 0-based index: sess[j] is wear number j + 1 in time order
                test_session = sess[j]
                train_sessions = sess[:j]                       # past wears only
                if (model_name, p_num, test_session) in done_keys:
                    continue
                if SMOKE and j != 1:
                    continue

                source = RUNG1[model_name][(p_num, test_session)]           # same participant, same test session
                assert source['participant_id'] == p_num and source['test_session'] == test_session
                params = source['best_params']

                held_mask = FEATURES['participant_id'] == p_num
                train_mask = held_mask & FEATURES['session_id'].isin(train_sessions)
                test_mask = held_mask & (FEATURES['session_id'] == test_session)

                # leak guards -- only wears strictly before the test wear, none after
                assert not (train_mask & test_mask).any()
                assert set(FEATURES.loc[train_mask, 'session_id'].unique()) == set(train_sessions)
                assert max(train_sessions) < test_session
                assert len(train_sessions) == j

                X_train = FEATURES.loc[train_mask, FEATURE_COLS].to_numpy()
                y_train = FEATURES.loc[train_mask, 'pct_mvc_corr'].to_numpy()
                X_test = FEATURES.loc[test_mask, FEATURE_COLS].to_numpy()
                y_test = FEATURES.loc[test_mask, 'pct_mvc_corr'].to_numpy()

                t0 = time.time()
                model = make_model(params)
                model.fit(X_train, y_train)
                fit_s = time.time() - t0
                y_pred = np.asarray(model.predict(X_test), dtype=float)

                result = evaluate(y_test, y_pred, FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])
                assert result['overall']['n'] == source['metrics']['overall']['n']       # same test frames as rung 1

                fold_results.append({
                    'model': model_name,
                    'participant_id': int(p_num),
                    'fold': source['fold'],
                    'test_session': int(test_session),
                    'params_from': {'participant_id': int(p_num), 'test_session': int(test_session), 'fold': source['fold']},
                    'best_params': params,
                    'n_train_wears': len(train_sessions),
                    'train_sessions': [int(s) for s in train_sessions],
                    'fit_s': round(fit_s, 1),
                    'metrics': result,
                })
                done_keys.add((model_name, p_num, test_session))
                _save_checkpoint(fold_results)

                print(f"{model_name:8s} p{p_num:02d} s{test_session:<2d} train={len(train_sessions)} wears ({fit_s:5.1f}s)  "
                      f"fwd={result['overall']['rmse']:6.2f}  rung1={source['metrics']['overall']['rmse']:6.2f}  "
                      f"[{len(fold_results)}/{n_total}, {(time.time() - t_loop) / 60:.1f} min]")

                del model, X_train, y_train, X_test, y_test, y_pred
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    print(f"done -- {len(fold_results)} folds in {OUT_PATH.name}")
else:
    print('RUN_LOOP is False -- forward loop skipped')

### Pooled %MVC RMSE over sessions 2 to K, per participant
Participants are rows. `forward` = this notebook. `rung1` = the stored rung-1 folds restricted to the same test sessions (2 to K), same frames. Pooled over frames, the same way as `05c`: `sqrt(sum(n * rmse^2) / sum(n))` over the per-session `n` and `rmse`, which equals the RMSE over all pooled frames. The table reads the JSON, so it also works after a restart. A participant appears only when all its forward folds are done.

In [ ]:
def pooled(n, rmse):
    n = np.asarray(n, dtype=float)
    rmse = np.asarray(rmse, dtype=float)
    return float(np.sqrt((n * rmse ** 2).sum() / n.sum()))


if OUT_PATH.exists():
    with open(OUT_PATH) as f:
        fwd = json.load(f)
else:
    fwd = []

pooled_tables = {}
for model_name in MODELS:
    fwd_m = {(r['participant_id'], r['test_session']): r for r in fwd if r['model'] == model_name}
    rows = []
    for p, sess in SESSIONS.items():
        test_sessions = sess[1:]
        if not all((p, s) in fwd_m for s in test_sessions):
            continue
        n_f = [fwd_m[(p, s)]['metrics']['overall']['n'] for s in test_sessions]
        n_r = [RUNG1[model_name][(p, s)]['metrics']['overall']['n'] for s in test_sessions]
        assert n_f == n_r                                                       # same frames in both
        assert all(fwd_m[(p, s)]['n_train_wears'] == i + 1 for i, s in enumerate(test_sessions))
        rows.append({
            'participant': f'p{p:03d}',
            'n_wears': len(sess),
            'rung1': pooled(n_r, [RUNG1[model_name][(p, s)]['metrics']['overall']['rmse'] for s in test_sessions]),
            'forward': pooled(n_f, [fwd_m[(p, s)]['metrics']['overall']['rmse'] for s in test_sessions]),
        })
    if not rows:
        continue
    table = pd.DataFrame(rows)
    table['fwd - rung1'] = table['forward'] - table['rung1']
    pooled_tables[model_name] = table
    print(model_name)
    print(table.round(2).to_string(index=False))
    print('mean', table[['rung1', 'forward', 'fwd - rung1']].mean().round(2).to_dict(),
          '| forward worse than rung1 in', int((table['forward'] > table['rung1']).sum()), 'of', len(table))
    print()

Per-wear table (plan decision 11). Participants are rows, test wear (session number) is the column. Wear 1 has no forward score. First block = forward RMSE, second block = rung-1 RMSE for the same wear. A missing session shows as NaN (participant 001 has no session 4, participant 010 has no session 6).

In [ ]:
for model_name in MODELS:
    fwd_m = {(r['participant_id'], r['test_session']): r['metrics']['overall']['rmse'] for r in fwd if r['model'] == model_name}
    if not fwd_m:
        continue
    cols = list(range(2, 11))
    fwd_tab = pd.DataFrame(
        {s: {f'p{p:03d}': fwd_m.get((p, s), np.nan) for p in SESSIONS} for s in cols})
    r1_tab = pd.DataFrame(
        {s: {f'p{p:03d}': (RUNG1[model_name][(p, s)]['metrics']['overall']['rmse'] if s in SESSIONS[p] else np.nan) for p in SESSIONS} for s in cols})
    print(model_name, '-- forward RMSE by test wear')
    print(fwd_tab.round(2).to_string())
    print(model_name, '-- rung-1 RMSE by test wear')
    print(r1_tab.round(2).to_string())
    print(model_name, '-- mean over participants: forward', fwd_tab.mean().round(2).to_dict())
    print(model_name, '--                          rung 1 ', r1_tab.mean().round(2).to_dict())
    print()